In [26]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

In [27]:
df = pd.read_csv("../outputs/final_dataset.csv")
df = df.dropna()
print(df["기준_년분기_코드"].value_counts().sort_index())

기준_년분기_코드
20201    8048
20202    8093
20203    8111
20204    8108
20211    8345
20212    8352
20213    8360
20214    8357
20221    8342
20222    8349
20223    8344
20224    8337
20231    8320
20232    8312
20233    8311
20234    8289
20241    8285
20242    8278
20243    8248
20244    8237
20251    8202
20252    8221
20253    8240
Name: count, dtype: int64


In [28]:
# 분기 순서 정렬 후 다음 분기 매출을 다음 행으로 shift
df = df.sort_values(["행정동코드", "통합카테고리", "기준_년분기_코드"])
df["다음분기_매출"] = df.groupby(["행정동코드", "통합카테고리"])["당월매출합"].shift(-1)

# label: 다음 분기 매출이 증가하면 1 (유망), 감소하면 0
df["label"] = (df["다음분기_매출"] > df["당월매출합"]).astype(int)

# 마지막 분기(Q3)는 다음 분기 없으므로 제외
labeled = df[df["다음분기_매출"].notna()].copy()

print("label 분포:")
print(labeled["label"].value_counts())
print(f"\n학습 가능 샘플 수: {len(labeled)}")

label 분포:
label
1    92848
0    88513
Name: count, dtype: int64

학습 가능 샘플 수: 181361


In [29]:
le = LabelEncoder()
labeled["업종_encoded"] = le.fit_transform(labeled["통합카테고리"])
labeled["분기번호"] = labeled["기준_년분기_코드"] % 10  # 계절성 반영 (1~4)

feature_cols = [
    "총유동인구", "유동_20대비율",
    "당월매출합", "매출_20대비율",
    "업종_점포당매출", "업종_매출점유율",
    "경쟁강도", "업종_포화도",
    "MZ_차이", "유동대비매출", "점포대비유동",
    "업종_encoded", "분기번호"
]

X = labeled[feature_cols]
y = labeled["label"]

all_quarters = sorted(df["기준_년분기_코드"].unique())
quarters = sorted(labeled["기준_년분기_코드"].unique())

# 시간 순서를 지키는 temporal split: 마지막 분기만 테스트, 나머지 전부 훈련
train_mask = labeled["기준_년분기_코드"] < quarters[-1]
test_mask  = labeled["기준_년분기_코드"] == quarters[-1]

X_train, y_train = X[train_mask], y[train_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

print(f"훈련: {len(X_train):,}개 ({quarters[0]} ~ {quarters[-2]})")
print(f"테스트: {len(X_test):,}개 ({quarters[-1]} → {all_quarters[-1]})")

훈련: 173,201개 (20201 ~ 20251)
테스트: 8,160개 (20252 → 20253)


In [30]:
model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",   # 성장/감소 비율 불균형 대응
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["감소", "성장"]))
print("AUC-ROC:", round(roc_auc_score(y_test, y_prob), 4))

# Feature 중요도
fi = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\n[Feature 중요도]")
print(fi.round(4))

              precision    recall  f1-score   support

          감소       0.54      0.79      0.64      3840
          성장       0.68      0.40      0.50      4320

    accuracy                           0.58      8160
   macro avg       0.61      0.59      0.57      8160
weighted avg       0.61      0.58      0.56      8160

AUC-ROC: 0.6574

[Feature 중요도]
분기번호          0.0890
업종_점포당매출      0.0844
업종_매출점유율      0.0839
업종_포화도        0.0826
MZ_차이         0.0822
유동대비매출        0.0785
유동_20대비율      0.0784
당월매출합         0.0784
점포대비유동        0.0783
매출_20대비율      0.0777
총유동인구         0.0756
경쟁강도          0.0591
업종_encoded    0.0519
dtype: float64


In [31]:
def recommend_dong_by_업종(업종명, df, model, le, feature_cols, top_n=10):
    """업종 클릭 → 창업하기 좋은 행정동 추천"""
    latest_q = df["기준_년분기_코드"].max()
    subset = df[(df["통합카테고리"] == 업종명) & (df["기준_년분기_코드"] == latest_q)].copy()

    if subset.empty:
        print(f"'{업종명}' 데이터가 없습니다.")
        return None

    subset["업종_encoded"] = le.transform(subset["통합카테고리"])
    subset["분기번호"] = subset["기준_년분기_코드"] % 10
    subset["성장확률"] = model.predict_proba(subset[feature_cols])[:, 1]

    result = (
        subset.nlargest(top_n, "성장확률")[["행정동명", "성장확률"]]
        .reset_index(drop=True)
    )
    result.index += 1

    print(f"\n[{업종명}] 창업 추천 행정동 TOP {top_n}\n")
    for i, row in result.iterrows():
        print(f"{i}위: {row['행정동명']} (성장확률: {row['성장확률']:.1%})")

    return result


def recommend_업종_by_dong(행정동명, df, model, le, feature_cols):
    """행정동 클릭 → 어울리는 업종 TOP 5 추천"""
    latest_q = df["기준_년분기_코드"].max()
    subset = df[(df["행정동명"] == 행정동명) & (df["기준_년분기_코드"] == latest_q)].copy()

    if subset.empty:
        print(f"'{행정동명}' 데이터가 없습니다.")
        return None

    subset["업종_encoded"] = le.transform(subset["통합카테고리"])
    subset["분기번호"] = subset["기준_년분기_코드"] % 10
    subset["성장확률"] = model.predict_proba(subset[feature_cols])[:, 1]

    result = (
        subset.nlargest(5, "성장확률")[["통합카테고리", "성장확률"]]
        .reset_index(drop=True)
    )
    result.index += 1

    print(f"\n[{행정동명}] 추천 업종 TOP 5\n")
    for i, row in result.iterrows():
        print(f"{i}위: {row['통합카테고리']} (성장확률: {row['성장확률']:.1%})")

    return result


# 테스트
print(df["통합카테고리"].unique())  # 카테고리명 확인
recommend_dong_by_업종("카페", df, model, le, feature_cols)
print()
recommend_업종_by_dong("역삼1동", df, model, le, feature_cols)

<StringArray>
['B2B 서비스',     '미용실',   '분식/간식',  '뷰티/화장품', '생활용품 소매',   '수리/세탁',   '식품 소매',
 '양식/기타외식',    '예술학원',   '의료/약국',   '의류/패션',    '일반학원',      '일식',   '전자/통신',
      '주점',      '중식',      '카페',     '편의점',      '한식',      '숙박',  '스포츠/레저',
   '오락/유흥',    '애완동물']
Length: 23, dtype: str

[카페] 창업 추천 행정동 TOP 10

1위: 안암동 (성장확률: 82.7%)
2위: 삼청동 (성장확률: 77.7%)
3위: 필동 (성장확률: 73.3%)
4위: 돈암1동 (성장확률: 73.3%)
5위: 회기동 (성장확률: 73.0%)
6위: 가락1동 (성장확률: 72.7%)
7위: 제기동 (성장확률: 70.7%)
8위: 명동 (성장확률: 70.3%)
9위: 사근동 (성장확률: 70.0%)
10위: 신촌동 (성장확률: 70.0%)


[역삼1동] 추천 업종 TOP 5

1위: 중식 (성장확률: 85.0%)
2위: 한식 (성장확률: 79.3%)
3위: 일식 (성장확률: 74.0%)
4위: 주점 (성장확률: 70.3%)
5위: 생활용품 소매 (성장확률: 69.7%)


,통합카테고리,성장확률
1,중식,0.850000
2,한식,0.793333
3,일식,0.740000
4,주점,0.703333
5,생활용품 소매,0.696667
